# Lab 9 — Semantic Caching and Cost Control

**Production Readiness Pack | CPU | `OPENAI_API_KEY` recommended (a scripted stand-in runs without one)**

---

The cheapest model call is the one you never make. If a hundred users ask the same thing, you should pay for one answer, not a hundred. That is caching, and for exact repeats it is old, boring and safe.

LLM apps want more than that. Users rarely type the same string twice, but they ask the same *question* in different words all day. A **semantic cache** embeds each question (MiniLM again, from Labs 0 and 6) and reuses a stored answer when a new question means nearly the same thing. It works, and it saves real money. It also creates a new way to be wrong: a fast, confident answer to a question nobody asked.

**Coming from Lab 8:** a trace tells you what one request did and what it cost. This lab tries to avoid paying for the next one.

## What you will walk out with

1. An exact cache and a semantic cache, each about fifteen lines, and the difference between them.
2. A false hit: the enterprise customer gets the premium customer's refund policy, in milliseconds.
3. Proof, from real similarity numbers, that no single threshold fixes that.
4. The fix that does work: putting who is asking into the cache key.
5. Actual dollars saved, from the token counts the API reports.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} openai sentence-transformers pandas python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret if you added one
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:                              # not on Colab, or no secret: try .env
    from dotenv import load_dotenv
    load_dotenv()

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
MODEL_NAME = "gpt-4o-mini"
print("OpenAI enabled:", USE_OPENAI, "" if USE_OPENAI else "(scripted stand-in instead)")

In [ ]:
import time
from dataclasses import dataclass
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

THRESHOLD   = 0.78              # one threshold for the whole notebook, as a real cache would have
DOC_VERSION = "faq-v1"          # bump when the FAQ changes
TTL_SECONDS = 60 * 60           # entries older than this are ignored

# gpt-4o-mini list prices, USD per 1M tokens, checked 2026-09-22. They change; check before you trust the totals.
PRICE_IN, PRICE_OUT = 0.15, 0.60

---

## 1. The model answers from an FAQ

The assistant answers from the five facts below, the same "answer only from this" pattern as Lab 6. That matters for this lab: because the answers are specific, a wrong cache hit will be *visibly* wrong, not just vaguely off.

The last two facts are the trap. Premium and enterprise customers have different refund terms.

In [ ]:
FAQ = """QLoRA fine-tunes small LoRA adapters on top of a frozen 4-bit quantized base model, which cuts GPU memory for training.
RAG retrieves relevant context at request time and puts it in the prompt so answers are grounded in your documents.
An OpenAI-compatible API lets the same client call different backends by changing base_url.
Premium plan: refunds are available within 30 days of purchase, no questions asked.
Enterprise plan: refund terms are set by the signed contract and handled by the account team; there is no self-service refund."""

SYSTEM = "Answer in one or two sentences, using only this FAQ. If it does not cover the question, say so.\n\n" + FAQ

`call_model` returns the answer *and* the token counts the API reports, so every cost in this lab is measured rather than guessed. Without a key, a scripted stand-in picks the matching FAQ line and pretends to take a second.

In [ ]:
from openai import OpenAI

def call_model(question):
    """Returns (answer, input_tokens, output_tokens)."""
    if not USE_OPENAI:
        time.sleep(1.0)
        q = question.lower()
        lines = FAQ.split("\n")
        pick = (lines[4] if "enterprise" in q else lines[3] if "refund" in q or "money back" in q
                else lines[0] if "qlora" in q else lines[1] if "rag" in q else lines[2])
        return pick, 250, 40
    r = OpenAI().chat.completions.create(model=MODEL_NAME, temperature=0,
                                         messages=[{"role": "system", "content": SYSTEM},
                                                   {"role": "user", "content": question}])
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens

def cost_usd(tokens_in, tokens_out):
    return (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000

answer, t_in, t_out = call_model("What is QLoRA?")
print(answer)
print(f"{t_in} tokens in, {t_out} out, ${cost_usd(t_in, t_out):.6f}")

**Checkpoint:** notice where the tokens go. Almost all of them are *input*: the FAQ rides along on every call. That is typical of RAG and system-prompt-heavy apps, and it is why caching them pays off.

---

## 2. Exact cache

A dictionary from question string to answer. Safe, since the same string gets the same answer, and nearly useless for chat: people do not retype questions character for character.

In [ ]:
exact_cache = {}

def ask_exact(question):
    t0 = time.perf_counter()
    if question in exact_cache:
        return {"question": question, "hit": True, "ms": round((time.perf_counter() - t0) * 1000, 1), "cost": 0.0}
    answer, t_in, t_out = call_model(question)
    exact_cache[question] = answer
    return {"question": question, "hit": False, "ms": round((time.perf_counter() - t0) * 1000, 1), "cost": cost_usd(t_in, t_out)}

pd.DataFrame([ask_exact(q) for q in ["How does QLoRA reduce GPU memory?",
                                      "How does QLoRA reduce GPU memory?",
                                      "Why does QLoRA need less GPU RAM?"]])

**Checkpoint:** the repeat is a free hit, answered in well under a millisecond. The paraphrase on the third row costs full price, even though any human would call it the same question.

---

## 3. Semantic cache

Store an embedding with each answer. On a new question, embed it, find the closest stored question, and reuse its answer if the similarity clears `THRESHOLD`.

A production entry carries more than question and answer. It records which model produced it, against which version of the FAQ, and when, so a model upgrade or an edited FAQ cannot serve stale answers. There is also a `scope` field. Ignore it for now; section 5 is about it.

In [ ]:
@dataclass
class CacheEntry:
    question: str
    answer: str
    embedding: np.ndarray
    scope: str            # who this answer is valid for (section 5)
    model_name: str
    doc_version: str
    created_at: float

cache = []
ledger = []               # every request, hit or miss, with what it cost

def embed(text):
    return embedder.encode(text, normalize_embeddings=True)

Lookup skips any entry that is stale, from another model or FAQ version, or from another scope. Then it keeps the most similar of what is left. Embeddings are normalized, so a dot product is the cosine similarity.

In [ ]:
def lookup(question, scope):
    q = embed(question)
    best, best_score = None, 0.0
    for e in cache:
        if e.model_name != MODEL_NAME or e.doc_version != DOC_VERSION or e.scope != scope:
            continue
        if time.time() - e.created_at > TTL_SECONDS:
            continue
        score = float(q @ e.embedding)
        if score > best_score:
            best, best_score = e, score
    return (best, best_score) if best_score >= THRESHOLD else (None, best_score)

In [ ]:
def ask(question, scope="public"):
    t0 = time.perf_counter()
    hit, score = lookup(question, scope)
    if hit:
        answer, cost, matched = hit.answer, 0.0, hit.question
        t_in, t_out = 0, 0
    else:
        answer, t_in, t_out = call_model(question)
        cost, matched = cost_usd(t_in, t_out), None
        cache.append(CacheEntry(question, answer, embed(question), scope, MODEL_NAME, DOC_VERSION, time.time()))
    # what this request would have cost without the cache: a hit saves the price of the call it reused
    full_cost = cost if not hit else next(r["cost"] for r in ledger if r["question"] == hit.question and not r["hit"])
    ledger.append({"question": question, "scope": scope, "hit": hit is not None, "similarity": round(score, 3),
                   "matched": matched, "ms": round((time.perf_counter() - t0) * 1000, 1),
                   "cost": cost, "cost_without_cache": full_cost, "answer": answer})
    return ledger[-1]

## 4. Paraphrases hit, and one gets missed

Three ways of asking the same QLoRA question. Before you run it: which of the second and third will clear 0.78?

In [ ]:
cache.clear(); ledger.clear()
for q in ["How does QLoRA reduce GPU memory?",
          "Why does QLoRA need less GPU RAM?",
          "Explain how QLoRA makes fine-tuning cheaper on GPUs."]:
    ask(q)

pd.DataFrame(ledger)[["question", "hit", "similarity", "ms", "cost"]]

**Checkpoint:** "GPU RAM" scores about 0.89 and hits, answered in a few milliseconds instead of a second or more. "Makes fine-tuning cheaper" scores about 0.76 and misses. A human hears the same question; the embedding hears a different one. You could lower the threshold to catch it. Hold that thought.

---

## 5. The false hit

A premium customer asks about refunds. Then an enterprise customer asks the same thing about *their* plan.

In [ ]:
cache.clear(); ledger.clear()
ask("What is the refund policy for the premium plan?")
ask("What is the refund policy for the enterprise plan?")

pd.DataFrame(ledger)[["question", "hit", "similarity", "matched", "answer"]]

The enterprise customer was told they can have a refund within 30 days, no questions asked. The FAQ says the opposite. The answer came back in milliseconds and cost nothing, which is exactly what makes a false hit dangerous: it looks like the cache working.

The obvious fix is a stricter threshold. Before you reach for it, look at what the threshold would have to separate.

In [ ]:
pairs = [
    ("How does QLoRA reduce GPU memory?",               "Why does QLoRA need less GPU RAM?",                         "yes"),
    ("How does QLoRA reduce GPU memory?",               "How does QLoRA reduce CPU memory?",                         "NO"),
    ("What is the refund policy for the premium plan?", "What is the refund policy for the premium plan in Germany?", "NO"),
    ("What is the refund policy for the premium plan?", "What is the refund policy for the enterprise plan?",        "NO"),
    ("How does QLoRA reduce GPU memory?",               "Explain how QLoRA makes fine-tuning cheaper on GPUs.",      "yes"),
    ("What is the refund policy for the premium plan?", "Can I get my money back on premium?",                       "yes"),
]
rows = [{"cached": a, "new question": b, "similarity": round(float(embed(a) @ embed(b)), 3), "same answer?": ok}
        for a, b, ok in pairs]
pd.DataFrame(rows).sort_values("similarity", ascending=False)

**Checkpoint:** read the "same answer?" column from top to bottom. The yes and NO rows are interleaved. A CPU question scores higher than a genuine paraphrase. "Can I get my money back on premium?", which should hit, scores below the enterprise question, which must not.

There is no line you can draw. Any threshold high enough to stop the enterprise false hit also misses real paraphrases, and it still lets the Germany question through. Embeddings measure how alike two sentences *sound*. Whether two questions deserve the same answer is a business rule, and the embedding has never seen your business rules.

---

## 6. The fix: scope the cache

Some facts about a request never appear in its wording: which plan the customer is on, which tenant they belong to, which country they are in, what they are allowed to see. In a real app they come from the logged-in session, not from the question. Put them in the cache key, and a premium answer can only ever be reused for another premium customer.

That is what `scope` does. Same two questions, same threshold, but now each carries the customer's plan.

In [ ]:
cache.clear(); ledger.clear()
ask("What is the refund policy for the premium plan?",    scope="premium")
ask("What is the refund policy for the enterprise plan?", scope="enterprise")
ask("Can I get my money back on premium?",                scope="premium")

pd.DataFrame(ledger)[["question", "scope", "hit", "similarity", "answer"]]

**Checkpoint:** the enterprise question misses, because there was nothing in its scope to reuse, and gets the correct answer from the model. The threshold did not change. The third question stayed in the premium scope and scored below 0.78, so it missed and paid full price. That is the safe way to be wrong.

Scoping is not free. Every scope has its own cold cache, so hit rates drop as scopes multiply. That is the real trade-off to tune, and it is a product decision, not an embedding one.

---

## 7. What it saved

Run a small, realistic mix through one scoped cache and add up the bill.

In [ ]:
cache.clear(); ledger.clear()
traffic = [
    ("How does QLoRA reduce GPU memory?", "public"),
    ("Why does QLoRA need less GPU RAM?", "public"),
    ("How does QLoRA cut GPU memory use?", "public"),
    ("What does RAG do?", "public"),
    ("What does RAG actually do?", "public"),
    ("What is the refund policy for the premium plan?", "premium"),
    ("What's the refund policy on the premium plan?", "premium"),
    ("What is the refund policy for the enterprise plan?", "enterprise"),
    ("How do I switch backends with the OpenAI client?", "public"),
    ("How do I switch backends using the OpenAI client?", "public"),
]
for q, s in traffic:
    ask(q, scope=s)

df = pd.DataFrame(ledger)
print(f"hit rate      : {df.hit.mean():.0%}")
print(f"paid          : ${df.cost.sum():.6f}")
print(f"without cache : ${df.cost_without_cache.sum():.6f}")
print(f"saved         : {1 - df.cost.sum() / df.cost_without_cache.sum():.0%}")
df[["question", "scope", "hit", "similarity", "ms"]]

**Checkpoint:** the dollar amounts are tiny because ten questions are tiny. Multiply by your traffic. At 10,000 questions a day, the saved percentage is what matters, and so is the latency column: every hit came back in milliseconds.

---

## 8. What is safe to cache

| Scenario | Cache it? | Why |
|---|---|---|
| Public FAQ: "What is QLoRA?" | Yes | Stable, same answer for everyone |
| Refund policy by plan | Yes, scoped by plan | Section 6 |
| Product docs for version 2.1 | Yes, if `doc_version` is in the key | The docs change between versions |
| "What is my invoice total?" | No | Personal data; one user's answer must never reach another |
| "Is the service down right now?" | No, or a very short TTL | The true answer changes by the minute |

| Gotcha | Why it matters | Mitigation |
|---|---|---|
| Similar wording, different policy | Section 5 | Scope by plan, tenant, region, permissions |
| Stale documents | A cached answer quotes last month's policy | Invalidate on `doc_version` or content hash |
| Model upgrade | Cached answers hide how the new model behaves | Model and prompt version in the entry |
| Invisible caching | You cannot tell a generated answer from a reused one | Log hit, similarity and matched question (Lab 8's trace) |

Start with an exact cache on safe public questions. Move to semantic caching only where you know what the scope is.

---

## Try it

1. Add "How does QLoRA reduce CPU memory?" to `traffic` in the public scope. What does it get back, and is that answer wrong?
2. Set `THRESHOLD = 0.70`, re-run from section 3, and count the new hits. Are any of them wrong?
3. Bump `DOC_VERSION` to `"faq-v2"` halfway through `traffic`. What happens to the hit rate, and why is that correct?
4. Pick one fact about *your* users that would have to be in the scope for your Capstone.

## What to take with you

1. **An exact cache is safe and rarely hits. A semantic cache hits often and can be wrong.**
2. **Similarity is not equivalence.** No threshold separates "sounds alike" from "has the same answer", as the table in section 5 showed.
3. **Scope is the real control.** Who is asking belongs in the key, not in the embedding.
4. **Every entry needs provenance:** model, document version, time. Otherwise upgrades and edits serve stale answers.
5. **Measure it in tokens.** The API tells you exactly what each call cost, so the savings are a number, not a hope.

## Next

[Lab 10 — Containerization](../10_Containerization/README.md) packages the Lab 5 proxy as a Docker image. It needs Docker on *your* machine; Colab can write the files but cannot build them.